For local pip install

In [ ]:
! pip install -U \
    langchain \
    langchain-core \
    langchain-community \
    langchain-groq \
    langchain-huggingface \
    sentence-transformers \
    faiss-cpu \
    python-dotenv

In [4]:
import warnings
warnings.filterwarnings


<function warnings.filterwarnings(action, message='', category=<class 'Warning'>, module='', lineno=0, append=False)>

Lib of pytthon

In [5]:
import os

from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

/opt/anaconda3/envs/jub/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/9f/00d3gq1s0w3_0m391myx76400000gn/T/ipykernel_94652/1190506230.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [6]:
# 1. Load the variables from the .env file into the environment
load_dotenv()

# 2. Access your key securely
api_key = os.getenv("GROQ_API_KEY")
#db_url = os.getenv("DATABASE_URL")

Load environment variables 

Create sample document 

In [19]:
documents = [
    Document(
        page_content=(
            "LangGraph is a framework for building stateful "
            "and multi-agent AI applications."
        ),
        metadata={"source": "langgraph_notes"},
    ),
       Document(
        page_content=(
            "Inception BD is a Edtech Platform "
            "and provides courses on AI."
        ),
        metadata={"source": "inception_notes"},
    ),
    Document(
        page_content=(
            "RAG stands for Retrieval-Augmented Generation. "
            "It retrieves relevant information before generating an answer."
        ),
        metadata={"source": "rag_notes"},
    ),
    Document(
        page_content=(
            "Groq provides fast inference for supported large "
            "language models through the Groq API."
        ),
        metadata={"source": "groq_notes"},
    ),
    Document(
        page_content=(
            "FAISS is a vector similarity-search library. "
            "It can retrieve documents whose embeddings are close "
            "to the query embedding."
        ),
        metadata={"source": "faiss_notes"},
    ),
]


In [8]:
documents

[Document(metadata={'source': 'langgraph_notes'}, page_content='LangGraph is a framework for building stateful and multi-agent AI applications.'),
 Document(metadata={'source': 'inception_notes'}, page_content='Inception BD is a Edtech Platform and provides courses on AI.'),
 Document(metadata={'source': 'rag_notes'}, page_content='RAG stands for Retrieval-Augmented Generation. It retrieves relevant information before generating an answer.'),
 Document(metadata={'source': 'groq_notes'}, page_content='Groq provides fast inference for supported large language models through the Groq API.'),
 Document(metadata={'source': 'faiss_notes'}, page_content='FAISS is a vector similarity-search library. It can retrieve documents whose embeddings are close to the query embedding.')]

In [9]:
type(documents[0])

langchain_core.documents.base.Document

Load the embedding model

In [20]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={
        "normalize_embeddings": True,
    },
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6007.86it/s]


Crate the FAISS vector store and store embedings 

In [21]:

vector_store = FAISS.from_documents(
    documents=documents,
    embedding=embeddings,
)


In [22]:
vector_store.save_local("faiss_index")

In [23]:
vector_store = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)

Create a Retriever

In [14]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2},
)

Initialize the Groq Model

In [33]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
    max_retries=2,
    api_key = 'GROQ_API_KEY'

)

In [25]:
response = llm.invoke("Hi Tell me about you")

In [26]:
response = llm.invoke('Hi tell me about')

In [27]:
response.content

'It looks like you were about to ask me something, but it got cut off. What would you like to know? I can tell you about a wide range of topics, from science and history to entertainment and culture.'

Create the prompt

In [28]:
prompt = ChatPromptTemplate.from_template(
    """
You are a helpful assistant.

Answer the question using only the provided context.

If the answer is not present in the context, say:
"I do not know based on the provided context."

Context:
{context}

Question:
{question}

Answer:
"""
)


Format retrieved documents

In [29]:
def format_documents(retrieved_documents: list[Document]) -> str:
    return "\n\n".join(
        document.page_content
        for document in retrieved_documents
    )



Build the RAG chain

In [30]:
rag_chain = (
    {
        "context": retriever | format_documents,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)


Ask question

In [31]:
def ask_question(question: str) -> str:
    if not question.strip():
        raise ValueError("Question cannot be empty.")

    return rag_chain.invoke(question)

In [34]:
if __name__ == "__main__":
    while True:
        user_question = input(
            "\nAsk a question or type 'exit': "
        ).strip()

        if user_question.lower() == "exit":
            print("Application closed.")
            break

        try:
            answer = ask_question(user_question)

            print("\nAnswer:")
            print(answer)

        except Exception as error:
            print(f"\nError: {error}")


Answer:
RAG stands for Retrieval-Augmented Generation.

Answer:
I do not know based on the provided context.

Answer:
I do not know based on the provided context.
Application closed.


''' RAG stands for Retrieval-Augmented Generation. "
            "It retrieves relevant information before generating an answer."  '''